# Mikaela Cashman's GTDB Metdata -> Environmental Matrix Processing

Our goal is to link each genome/sample in Mikaela's GTDB data which are all genomes/MAGS linked to biosamples to as much environmental data as possible. We want the closest relevant information to be found. So the challenge is the smallest bounding box that has data for each relevant layer that we can link to the point. 

## Load and process Mikaela's data

In [ ]:
import pandas as pd
import re

def find_nonconforming_latlon(df, column):
    """
    Return rows from df[column] that do not match the expected lat/lon format:
    '<decimal> N|S <decimal> E|W'
    """
    # Regex: number + space + hemisphere + space + number + space + hemisphere
    pattern = re.compile(r'^\s*-?\d+(\.\d+)?\s+[NS]\s+-?\d+(\.\d+)?\s+[EW]\s*$', re.IGNORECASE)
    
    mask = ~df[column].astype(str).str.match(pattern)
    return df[mask]

In [ ]:
import re
import math
import numpy as np
import pandas as pd
from pyproj import CRS, Transformer

def dms_to_decimal(d, m=0, s=0, hemi=None):
    """Convert DMS to decimal degrees, applying hemisphere if given."""
    val = float(d) + float(m)/60 + float(s)/3600
    if hemi and hemi.upper() in ["S", "W"]:
        val = -val
    return val

def parse_latlon_value(s: str, utm_zone=None, utm_hemisphere="N"):
    """
    Parse a latitude/longitude string into decimal degrees.
    Supports:
      - Decimal pairs (with / , space)
      - Decimal with hemisphere
      - DMS with °′″ or compact with hemisphere suffix
      - UTM (if zone is given)
    Returns (lat, lon) or (NaN, NaN).
    """
    if pd.isna(s):
        return (np.nan, np.nan)
    
    s = str(s).strip()
    
    # Case 1: simple decimal pair with / , or space separator
    m = re.match(r'^\s*(-?\d+(?:\.\d+)?)\s*[,/ ]\s*(-?\d+(?:\.\d+)?)\s*$', s)
    if m:
        return float(m.group(1)), float(m.group(2))
    
    # Case 2: decimal + hemisphere (e.g. "40.7 N 74.0 W")
    m = re.match(r'^\s*(-?\d+(?:\.\d+)?)\s*([NS])\s+(-?\d+(?:\.\d+)?)\s*([EW])\s*$', s, re.I)
    if m:
        lat = float(m.group(1))
        if m.group(2).upper() == "S": lat = -lat
        lon = float(m.group(3))
        if m.group(4).upper() == "W": lon = -lon
        return lat, lon
    
    # Case 3: DMS with degree symbols (44°29′37″ N, 11°20′19″ E)
    # --- NEW robust DMS extraction (deals with embedded coords ---
    dms_pattern = re.compile(
        r'(\d+)[°º\s]+(\d+)?[\'′]?\s*(\d+(?:\.\d+)?)?["″\']?\s*([NSEW])',
        re.I
    )
    parts = dms_pattern.findall(s)
    if len(parts) >= 2:
        def convert(p):
            d, m, sec, hemi = p
            return dms_to_decimal(d, m or 0, sec or 0, hemi)
        lat = convert(parts[0])
        lon = convert(parts[1])
        return lat, lon
    
    # Case 4: Compact DMS with hemisphere suffix (e.g. "45°46'50.3N 4°52'11.2E")
    dms_compact = re.findall(r'(\d+)[°º]([\d\.]*)\'?([\d\.]*)?"?([NSEW])', s, re.I)
    if len(dms_compact) >= 2:
        def part_to_dec(d, m, s, h):
            return dms_to_decimal(d, m or 0, s or 0, h)
        lat = part_to_dec(*dms_compact[0])
        lon = part_to_dec(*dms_compact[1])
        return lat, lon
    
    # Case 5: UTM-like values (two big numbers with 'm')
    if "m" in s.lower() and utm_zone is not None:
        nums = re.findall(r'(\d+(?:[.,]\d+)*)', s)
        if len(nums) >= 2:
            northing = float(nums[0].replace(",", "."))
            easting = float(nums[1].replace(",", "."))
            crs_utm = CRS.from_proj4(f"+proj=utm +zone={utm_zone} +{'south' if utm_hemisphere=='S' else 'north'} +datum=WGS84 +units=m +no_defs")
            transformer = Transformer.from_crs(crs_utm, CRS.from_epsg(4326), always_xy=True)
            lon, lat = transformer.transform(easting, northing)
            return lat, lon
    
    return (np.nan, np.nan)

def parse_latlon_column(df, column, utm_zone=None, utm_hemisphere="N"):
    """Parse a pandas column of lat/lon strings into two new float columns."""
    parsed = df[column].apply(lambda s: parse_latlon_value(s, utm_zone=utm_zone, utm_hemisphere=utm_hemisphere))
    df["lat"] = parsed.apply(lambda x: x[0])
    df["lon"] = parsed.apply(lambda x: x[1])
    return df

In [ ]:
pwd

In [ ]:
import pandas as pd

df_gtdb= pd.read_csv('date_and_latlon_samples_extended.tsv', sep='\t',encoding='utf8')

In [ ]:
find_nonconforming_latlon(df_gtdb,'lat_lon')['lat_lon'].unique()

In [ ]:
df_gtdb=parse_latlon_column(df_gtdb, "lat_lon", utm_zone=33, utm_hemisphere="N")

In [ ]:
import folium
from folium.plugins import MarkerCluster

def plot_clustered_map(df, lat_col="lat", lon_col="lon", popup_col=None,
                       start_location=None, zoom_start=2, tiles="OpenStreetMap"):
    """
    Create a Folium map with clustered markers from a DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing latitude/longitude columns.
    lat_col : str
        Column name for latitude.
    lon_col : str
        Column name for longitude.
    popup_col : str or None
        Optional column to show in popup text when clicking markers.
    start_location : (lat, lon) or None
        If None, will use the mean of all coordinates.
    zoom_start : int
        Initial zoom level.
    tiles : str
        Basemap tiles (e.g., "OpenStreetMap", "CartoDB positron", "Stamen Terrain").
    """
    
    # Drop rows with missing coords
    df_clean = df.dropna(subset=[lat_col, lon_col])
    
    # Default center = mean lat/lon if not provided
    if start_location is None and not df_clean.empty:
        start_location = [df_clean[lat_col].mean(), df_clean[lon_col].mean()]
    elif start_location is None:
        start_location = [0, 0]
    
    # Create map
    m = folium.Map(location=start_location, zoom_start=zoom_start, tiles=tiles)
    
    # Add clustered markers
    marker_cluster = MarkerCluster().add_to(m)
    
    for _, row in df_clean.iterrows():
        lat, lon = row[lat_col], row[lon_col]
        popup_text = None
        if popup_col and popup_col in df_clean.columns:
            popup_text = str(row[popup_col])
        folium.Marker(location=[lat, lon], popup=popup_text).add_to(marker_cluster)
    
    return m


In [ ]:
m = plot_clustered_map(df_gtdb, lat_col="lat", lon_col="lon", popup_col="species")
#m.save("clustered_map.html")  # optional: save to file
m

In [ ]:
df_gtdb.columns

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm

def classify_environment_osm(df, lat_col="lat", lon_col="lon", radius=100):
    """
    Classify coordinates as 'free_environment' or 'other' using OSM Overpass API.
    Detects nearby buildings, universities, labs, or industrial sites.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Must contain lat_col and lon_col
    lat_col, lon_col : str
        Column names for latitude and longitude
    radius : int
        Search radius in meters around the point
    
    Returns
    -------
    pandas.DataFrame with new columns:
        - env_class: 'free_environment' or 'other'
        - env_tags: OSM feature tags (if any)
    """
    
    overpass_url = "http://overpass-api.de/api/interpreter"
    
    def query_osm(lat, lon, radius):
        query = f"""
        [out:json];
        (
          node(around:{radius},{lat},{lon})[building];
          way(around:{radius},{lat},{lon})[building];
          relation(around:{radius},{lat},{lon})[building];
          node(around:{radius},{lat},{lon})[amenity=university];
          way(around:{radius},{lat},{lon})[amenity=university];
          node(around:{radius},{lat},{lon})[industrial];
          way(around:{radius},{lat},{lon})[industrial];
        );
        out body;
        """
        try:
            r = requests.get(overpass_url, params={'data': query}, timeout=30)
            r.raise_for_status()
            data = r.json()
            if data.get("elements"):
                tags = []
                for el in data["elements"]:
                    if "tags" in el:
                        tags.append(el["tags"])
                return "other", tags
            else:
                return "free_environment", []
        except Exception as e:
            return "error", [{"error": str(e)}]
    
    # Get unique lat/lon pairs
    unique_coords = df[[lat_col, lon_col]].dropna().drop_duplicates()
    
    # Run OSM queries
    results = {}
    for _, row in tqdm(unique_coords.iterrows(), total=len(unique_coords), desc="OSM lookups"):
        lat, lon = row[lat_col], row[lon_col]
        env_class, tags = query_osm(lat, lon, radius)
        results[(lat, lon)] = (env_class, tags)
    
    # Map results back to original DataFrame
    df["env_class"] = df[[lat_col, lon_col]].apply(
        lambda x: results.get((x[lat_col], x[lon_col]), ("unknown", []))[0],
        axis=1
    )
    df["env_tags"] = df[[lat_col, lon_col]].apply(
        lambda x: results.get((x[lat_col], x[lon_col]), ("unknown", []))[1],
        axis=1
    )
    
    return df


In [ ]:
df_gtdb = classify_environment_osm(df_gtdb, lat_col="lat", lon_col="lon", radius=200)

In [ ]:
df_gtdb.to_csv('df_gtdb_tagged.tsv',sep='\t',encoding='utf8')

In [ ]:
df_gtdb

In [ ]:
df_gtdb['env_class'].value_counts()

In [ ]:
df_gtdb[df_gtdb['env_class']=='unknown']['lat'].unique()

In [ ]:
df_gtdb[df_gtdb['env_class']=='error']['env_broad_med_local'].unique()

In [ ]:
# subset the rows with errors
subset = df_gtdb[df_gtdb['env_class'] == 'error'].copy()

# re-run classification
subset_classified = classify_environment_osm(subset, lat_col="lat", lon_col="lon", radius=200)

# merge results back on index
df_gtdb.update(subset_classified)


I repeated ran the above cell until I was left with two errors that would not fix. 

In [ ]:
df_gtdb[df_gtdb['env_broad_med_local']=='marine biome [ENVO_00000447] | marine sediment [ENVO:03000033] | marine sediment [ENVO:03000033]']

That is not a good record.. the lat_lon has what looks perhaps like a bounding box so the extracted location is corrupted. That said... it IS a free environment

In [ ]:
mask = df_gtdb['env_broad_med_local'] == 'marine biome [ENVO_00000447] | marine sediment [ENVO:03000033] | marine sediment [ENVO:03000033]'
df_gtdb.loc[mask, 'env_class'] = 'free_environment'

Let's find all extracted lat lons that look odd. 

In [ ]:
import pandas as pd
import numpy as np

def valid_latlon(df, lat_col="lat", lon_col="lon"):
    """
    Returns a boolean Series indicating whether each row has a valid lat/lon pair
    in decimal degrees. Rows where both lat and lon are NaN are considered valid.
    """
    lat = df[lat_col]
    lon = df[lon_col]
    
    # mask for rows that are completely NaN → accept
    both_nan = lat.isna() & lon.isna()
    
    # mask for numeric and within valid ranges
    numeric = lat.apply(lambda x: isinstance(x, (int, float, np.number))) & \
              lon.apply(lambda x: isinstance(x, (int, float, np.number)))
    
    in_range = lat.between(-90, 90, inclusive="both") & lon.between(-180, 180, inclusive="both")
    
    return both_nan | (numeric & in_range)

In [ ]:
df_gtdb["latlon_valid"] = valid_latlon(df_gtdb)

In [ ]:
df_gtdb[df_gtdb['latlon_valid']==False]

In [ ]:
mask = df_gtdb['env_broad_med_local'] == 'marine biome [ENVO_00000447] | marine sediment [ENVO:03000033] | marine sediment [ENVO:03000033]'
df_gtdb.loc[mask, 'lat'] = 27.0388 
df_gtdb.loc[mask, 'lon'] = 111.2456

In [ ]:
list(df_gtdb['date'].unique())

In [ ]:
df_gtdb.to_csv('df_gtdb_tagged_cleaneed.tsv',sep='\t',encoding='utf8')

## Now find environmental data at reasonable coordinates. 

In [ ]:
# Load the cleaned dataset with env_class annotations
import pandas as pd
import numpy as np
import sys
import os

# Add env-agents to path
sys.path.insert(0, '..')

df_gtdb = pd.read_csv('df_gtdb_tagged_cleaneed.tsv', sep='\t', encoding='utf8', index_col=0)

print(f"📊 Dataset loaded: {df_gtdb.shape}")
print(f"🌍 Total samples: {len(df_gtdb):,}")
print(f"🆓 Free environment samples: {(df_gtdb['env_class'] == 'free_environment').sum():,}")
print(f"📍 Valid lat/lon pairs: {df_gtdb['latlon_valid'].sum():,}")

# Focus on free environment samples with valid coordinates
target_samples = df_gtdb[
    (df_gtdb['env_class'] == 'free_environment') & 
    (df_gtdb['latlon_valid'] == True) &
    (df_gtdb['lat'].notna()) & 
    (df_gtdb['lon'].notna())
].copy()

print(f"🎯 Target samples for environmental enrichment: {len(target_samples):,}")

# Get unique lat/lon pairs to minimize API calls
unique_coords = target_samples[['lat', 'lon', 'date']].drop_duplicates()
print(f"📍 Unique coordinate pairs: {len(unique_coords):,}")

target_samples.head()

In [ ]:
# Environmental Service Selection and Capabilities Discovery
from env_agents.core.models import RequestSpec, Geometry
from env_agents.adapters import CANONICAL_SERVICES
import pandas as pd
from IPython.display import display, HTML

def display_service_capabilities():
    """Display aesthetic overview of available environmental services for pangenome analysis"""
    
    # Priority services for pangenome environment linkage
    priority_services = {
        'NASA_POWER': '🌡️ Climate & Solar',
        'SoilGrids': '🌱 Soil Properties', 
        'WQP': '💧 Water Quality',
        'OpenAQ': '💨 Air Quality',
        'GBIF': '🦋 Biodiversity',
        'EARTH_ENGINE': '🛰️ Satellite Data',
        'USGS_NWIS': '🏔️ Hydrology',
        'EPA_AQS': '🏭 EPA Air Monitoring',
        'SSURGO': '🌾 Soil Survey',
        'OSM_Overpass': '🗺️ Geographic Features'
    }
    
    print("🌍 ENV-AGENTS ENVIRONMENTAL DATA SERVICES")
    print("=" * 55)
    print("Selected services optimal for pangenome-environment linkage:\n")
    
    service_details = []
    
    for service_name, description in priority_services.items():
        if service_name in CANONICAL_SERVICES:
            try:
                adapter_class = CANONICAL_SERVICES[service_name]
                adapter = adapter_class() if service_name != "EARTH_ENGINE" else adapter_class(asset_id="MODIS/061/MOD13Q1")
                caps = adapter.capabilities()
                
                # Extract key info
                variables = len(caps.get('variables', []))
                auth = "🔑 API Key" if caps.get('requires_auth', False) else "🆓 No Auth"
                coverage = caps.get('spatial_coverage', 'Global')
                
                service_details.append({
                    'Service': f"{description} **{service_name}**",
                    'Variables': f"{variables:,}",
                    'Auth': auth,
                    'Coverage': coverage
                })
                
                print(f"{description} **{service_name}**")
                print(f"   Variables: {variables:,} | {auth} | Coverage: {coverage}")
                
            except Exception as e:
                print(f"{description} **{service_name}** - ⚠️ Setup required")
        else:
            print(f"{description} **{service_name}** - ❌ Not available")
        
        print()
    
    print("🎯 **STRATEGY FOR 32K+ LOCATIONS:**")
    print("• Spatial clustering to minimize API calls")
    print("• Temporal matching with sample collection dates")
    print("• Progressive enrichment by data type priority")
    print("• Efficient storage with spatial indexing")
    
    return service_details

# Display capabilities
service_info = display_service_capabilities()

In [ ]:
# Spatial Clustering Optimization for API Efficiency
from sklearn.cluster import DBSCAN
import folium
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
import numpy as np

def optimize_spatial_clusters(coords_df, eps_deg=0.1, min_samples=2, max_cluster_size=100):
    """
    Cluster nearby coordinates to optimize API calls while respecting service limits.
    
    Parameters:
    -----------
    coords_df : DataFrame with 'lat', 'lon' columns
    eps_deg : float - maximum distance between points in cluster (degrees)
    min_samples : int - minimum points to form cluster
    max_cluster_size : int - maximum points per cluster (for API limits)
    
    Returns:
    --------
    DataFrame with cluster assignments and bounding boxes
    """
    
    # DBSCAN clustering in geographic coordinates
    coords_array = coords_df[['lat', 'lon']].values
    
    # Use DBSCAN with geographic distance approximation
    clustering = DBSCAN(eps=eps_deg, min_samples=min_samples).fit(coords_array)
    
    coords_df = coords_df.copy()
    coords_df['cluster_id'] = clustering.labels_
    
    # Split large clusters
    final_clusters = []
    cluster_id = 0
    
    for cluster_label in coords_df['cluster_id'].unique():
        if cluster_label == -1:  # Noise points get individual clusters
            noise_points = coords_df[coords_df['cluster_id'] == cluster_label]
            for idx, row in noise_points.iterrows():
                final_clusters.append({
                    'cluster_id': cluster_id,
                    'points': [idx],
                    'bbox': [row['lat'], row['lon'], row['lat'], row['lon']],
                    'center_lat': row['lat'],
                    'center_lon': row['lon'],
                    'point_count': 1
                })
                cluster_id += 1
        else:
            cluster_points = coords_df[coords_df['cluster_id'] == cluster_label]
            
            # Split large clusters
            if len(cluster_points) > max_cluster_size:
                # Sub-cluster large groups
                sub_coords = cluster_points[['lat', 'lon']].values
                sub_clustering = DBSCAN(eps=eps_deg/2, min_samples=min_samples).fit(sub_coords)
                
                for sub_label in np.unique(sub_clustering.labels_):
                    sub_mask = sub_clustering.labels_ == sub_label
                    sub_points = cluster_points.iloc[sub_mask]
                    
                    if len(sub_points) > 0:
                        bbox = [
                            sub_points['lat'].min(), sub_points['lon'].min(),
                            sub_points['lat'].max(), sub_points['lon'].max()
                        ]
                        
                        final_clusters.append({
                            'cluster_id': cluster_id,
                            'points': sub_points.index.tolist(),
                            'bbox': bbox,
                            'center_lat': sub_points['lat'].mean(),
                            'center_lon': sub_points['lon'].mean(),
                            'point_count': len(sub_points)
                        })
                        cluster_id += 1
            else:
                # Keep cluster as-is
                bbox = [
                    cluster_points['lat'].min(), cluster_points['lon'].min(),
                    cluster_points['lat'].max(), cluster_points['lon'].max()
                ]
                
                final_clusters.append({
                    'cluster_id': cluster_id,
                    'points': cluster_points.index.tolist(),
                    'bbox': bbox,
                    'center_lat': cluster_points['lat'].mean(),
                    'center_lon': cluster_points['lon'].mean(),
                    'point_count': len(cluster_points)
                })
                cluster_id += 1
    
    return pd.DataFrame(final_clusters)

# Apply clustering optimization
print("🔍 Optimizing spatial clusters for API efficiency...")
print(f"Processing {len(unique_coords):,} unique coordinate pairs")

# Cluster optimization
clusters_df = optimize_spatial_clusters(
    unique_coords.reset_index(drop=True),
    eps_deg=0.05,  # ~5km at equator
    min_samples=2,
    max_cluster_size=50  # Respect API limits
)

print(f"📊 Optimization results:")
print(f"   • Created {len(clusters_df)} spatial clusters")
print(f"   • Average cluster size: {clusters_df['point_count'].mean():.1f} points")
print(f"   • Largest cluster: {clusters_df['point_count'].max()} points")
print(f"   • Single-point clusters: {(clusters_df['point_count'] == 1).sum()}")
print(f"   • API call reduction: {len(unique_coords)} → {len(clusters_df)} ({100*(1-len(clusters_df)/len(unique_coords)):.1f}% reduction)")

clusters_df.head()

In [ ]:
# Visualize Spatial Clusters on Map
def visualize_clusters(clusters_df, unique_coords, sample_size=1000):
    """Create interactive map showing spatial clusters and optimization"""
    
    # Sample for visualization if too many points
    if len(clusters_df) > sample_size:
        display_clusters = clusters_df.sample(sample_size, random_state=42)
        print(f"📍 Displaying {sample_size} representative clusters (of {len(clusters_df)} total)")
    else:
        display_clusters = clusters_df
        print(f"📍 Displaying all {len(display_clusters)} clusters")
    
    # Create map centered on data
    center_lat = display_clusters['center_lat'].mean()
    center_lon = display_clusters['center_lon'].mean()
    
    m = folium.Map(location=[center_lat, center_lon], zoom_start=2, tiles='OpenStreetMap')
    
    # Add clusters with different colors by size
    for _, cluster in display_clusters.iterrows():
        # Color by cluster size
        if cluster['point_count'] == 1:
            color = 'blue'
            popup_text = f"Single point cluster"
        elif cluster['point_count'] <= 5:
            color = 'green'  
            popup_text = f"Small cluster: {cluster['point_count']} points"
        elif cluster['point_count'] <= 20:
            color = 'orange'
            popup_text = f"Medium cluster: {cluster['point_count']} points"
        else:
            color = 'red'
            popup_text = f"Large cluster: {cluster['point_count']} points"
        
        # Add cluster center marker
        folium.CircleMarker(
            location=[cluster['center_lat'], cluster['center_lon']],
            radius=min(3 + cluster['point_count']/5, 15),
            color=color,
            fillColor=color,
            fillOpacity=0.6,
            popup=popup_text
        ).add_to(m)
        
        # Add bounding box for multi-point clusters
        if cluster['point_count'] > 1:
            bbox = cluster['bbox']
            folium.Rectangle(
                bounds=[[bbox[0], bbox[1]], [bbox[2], bbox[3]]],
                color=color,
                weight=1,
                fillOpacity=0.1
            ).add_to(m)
    
    print(f"🗺️ Interactive map created with cluster visualization")
    print(f"   • Blue: Single points")
    print(f"   • Green: Small clusters (2-5 points)")  
    print(f"   • Orange: Medium clusters (6-20 points)")
    print(f"   • Red: Large clusters (21+ points)")
    
    return m

# Create cluster visualization
cluster_map = visualize_clusters(clusters_df, unique_coords)
cluster_map

In [ ]:
# Test Data Acquisition Pipeline - Starting with High-Value Services
import datetime
import pickle
import os
from pathlib import Path

def parse_date_range(date_str):
    """Parse various date formats from GTDB data into start/end dates for API queries"""
    if pd.isna(date_str):
        # Default to a reasonable range for environmental data
        return ("2020-01-01", "2021-01-01")
    
    date_str = str(date_str).strip()
    
    # Handle ranges (YYYY/YYYY or YYYY-MM-DD/YYYY-MM-DD)
    if '/' in date_str:
        parts = date_str.split('/')
        start_date = parts[0].strip()
        end_date = parts[1].strip() if len(parts) > 1 else start_date
    else:
        start_date = end_date = date_str
    
    # Standardize formats
    def standardize_date(d):
        if len(d) == 4:  # Just year
            return f"{d}-01-01"
        elif len(d) == 7:  # Year-month
            return f"{d}-01"
        else:
            return d
    
    start_date = standardize_date(start_date)
    end_date = standardize_date(end_date)
    
    # Ensure end_date is after start_date
    if end_date <= start_date:
        try:
            end_year = int(end_date[:4]) + 1
            end_date = f"{end_year}-01-01"
        except:
            end_date = start_date
    
    return (start_date, end_date)

def test_service_acquisition(service_name, clusters_sample, max_test_clusters=5):
    """
    Test environmental data acquisition for a specific service
    
    Parameters:
    -----------
    service_name : str - Name of service to test
    clusters_sample : DataFrame - Sample of clusters to test
    max_test_clusters : int - Maximum clusters to test (for safety)
    """
    
    print(f"🧪 TESTING {service_name} DATA ACQUISITION")
    print(f"=" * 50)
    
    if service_name not in CANONICAL_SERVICES:
        print(f"❌ {service_name} not available in CANONICAL_SERVICES")
        return None
    
    # Initialize adapter
    try:
        adapter_class = CANONICAL_SERVICES[service_name]
        if service_name == "EARTH_ENGINE":
            adapter = adapter_class(asset_id="MODIS/061/MOD13Q1")  # MODIS vegetation indices
        else:
            adapter = adapter_class()
            
        print(f"✅ {service_name} adapter initialized")
        
        # Show capabilities
        caps = adapter.capabilities()
        print(f"📊 Available variables: {len(caps.get('variables', []))}\")\n")
        
    except Exception as e:
        print(f"❌ Failed to initialize {service_name}: {str(e)}")
        return None
    
    # Test on sample clusters
    test_clusters = clusters_sample.head(max_test_clusters)
    results = []
    
    for idx, cluster in test_clusters.iterrows():
        print(f"📍 Testing cluster {cluster['cluster_id']} ({cluster['point_count']} points)")
        
        try:
            # Create bounding box geometry
            bbox = cluster['bbox']
            geometry = Geometry(
                type="bbox", 
                coordinates=[bbox[1], bbox[0], bbox[3], bbox[2]]  # [min_lon, min_lat, max_lon, max_lat]
            )
            
            # Use a standard time range for testing
            time_range = ("2020-01-01", "2020-12-31")
            
            # Create request
            spec = RequestSpec(
                geometry=geometry,
                time_range=time_range,
                variables=None,  # Get all available
                extra={"timeout": 30}
            )
            
            # Fetch data
            start_time = datetime.datetime.now()
            result_df = adapter.fetch(spec)
            elapsed = (datetime.datetime.now() - start_time).total_seconds()
            
            if len(result_df) > 0:
                print(f"   ✅ Success: {len(result_df)} observations in {elapsed:.1f}s")
                print(f"   🔬 Variables: {result_df['variable'].nunique()}")
                print(f"   📅 Date range: {result_df['time'].min()} to {result_df['time'].max()}")
                
                results.append({
                    'cluster_id': cluster['cluster_id'],
                    'point_count': cluster['point_count'],
                    'obs_count': len(result_df),
                    'variables': result_df['variable'].nunique(),
                    'time_range': (result_df['time'].min(), result_df['time'].max()),
                    'elapsed_seconds': elapsed,
                    'status': 'success',
                    'sample_data': result_df.head(3).to_dict('records')  # Store sample
                })
            else:
                print(f"   ⚠️  No data returned")
                results.append({
                    'cluster_id': cluster['cluster_id'],
                    'point_count': cluster['point_count'], 
                    'obs_count': 0,
                    'variables': 0,
                    'elapsed_seconds': elapsed,
                    'status': 'no_data'
                })
                
        except Exception as e:
            print(f"   ❌ Error: {str(e)[:100]}...")
            results.append({
                'cluster_id': cluster['cluster_id'],
                'point_count': cluster['point_count'],
                'status': 'error',
                'error': str(e)
            })
        
        print()
    
    # Summary
    successful = sum(1 for r in results if r['status'] == 'success')
    total_obs = sum(r.get('obs_count', 0) for r in results)
    
    print(f"📊 TEST SUMMARY FOR {service_name}:")
    print(f"   ✅ Successful queries: {successful}/{len(results)}")
    print(f"   📈 Total observations: {total_obs:,}")
    print(f"   ⚡ Average time per query: {np.mean([r.get('elapsed_seconds', 0) for r in results]):.1f}s")
    
    return results

# Test NASA_POWER (climate/solar data) - No authentication required
print("🚀 Starting test data acquisition pipeline...")
print(f"Testing with {len(clusters_df)} optimized clusters\\n")

# Select representative test clusters (mix of sizes)
small_clusters = clusters_df[clusters_df['point_count'] == 1].sample(min(2, len(clusters_df[clusters_df['point_count'] == 1])))
medium_clusters = clusters_df[(clusters_df['point_count'] > 1) & (clusters_df['point_count'] <= 10)].sample(min(2, len(clusters_df[(clusters_df['point_count'] > 1) & (clusters_df['point_count'] <= 10)])))
large_clusters = clusters_df[clusters_df['point_count'] > 10].sample(min(1, len(clusters_df[clusters_df['point_count'] > 10])))

test_sample = pd.concat([small_clusters, medium_clusters, large_clusters], ignore_index=True)
print(f"📋 Selected {len(test_sample)} test clusters representing different sizes\\n")

# Test NASA POWER first (reliable, no auth required)
nasa_results = test_service_acquisition('NASA_POWER', test_sample, max_test_clusters=5)

In [ ]:
# Export data for production script
import pandas as pd
from pathlib import Path

print("📤 EXPORTING DATA FOR PRODUCTION SCRIPT")
print("=" * 60)

# Export clusters
clusters_export = clusters_df.copy()
clusters_export['bbox'] = clusters_export['bbox'].apply(str)  # Convert lists to strings for CSV
output_clusters = Path('clusters_optimized.csv')
clusters_export.to_csv(output_clusters, index=False)
print(f"✅ Exported {len(clusters_export):,} spatial clusters to: {output_clusters}")

# Export genome samples (already have the TSV)
samples_file = Path('df_gtdb_tagged_cleaneed.tsv')
if samples_file.exists():
    print(f"✅ Genome samples ready at: {samples_file}")
    print(f"   ({len(target_samples):,} free environment samples)")
else:
    print(f"⚠️  Genome samples file not found, creating...")
    target_samples.to_csv(samples_file, sep='\t', encoding='utf8')
    print(f"✅ Exported genome samples to: {samples_file}")

print(f"\n🚀 READY TO RUN PRODUCTION SCRIPT")
print(f"\n{'='*60}")
print(f"PARALLEL EXECUTION STRATEGY")
print(f"{'='*60}")

print(f"\n💡 YES, you can run Phase 1 and Phase 2 in PARALLEL!")
print(f"   SQLite supports concurrent writes with proper locking.")

print(f"\n📋 OPTION 1: Run phases in parallel (fastest - ~62.7 hours total)")
print(f"\n   Terminal 1 (Phase 1 - 3 services):")
print(f"   python scripts/acquire_environmental_data.py \\")
print(f"       --phase 1 \\")
print(f"       --clusters {output_clusters} \\")
print(f"       --samples {samples_file}")

print(f"\n   Terminal 2 (Phase 2 - embeddings):")
print(f"   python scripts/acquire_environmental_data.py \\")
print(f"       --phase 2 \\")
print(f"       --clusters {output_clusters} \\")
print(f"       --samples {samples_file}")

print(f"\n📋 OPTION 2: Run individual services in parallel (maximum parallelization)")
print(f"\n   Terminal 1: python scripts/acquire_environmental_data.py --service NASA_POWER --clusters ... --samples ...")
print(f"   Terminal 2: python scripts/acquire_environmental_data.py --service SRTM --clusters ... --samples ...")
print(f"   Terminal 3: python scripts/acquire_environmental_data.py --service MODIS_NDVI --clusters ... --samples ...")
print(f"   Terminal 4: python scripts/acquire_environmental_data.py --service GOOGLE_EMBEDDINGS --clusters ... --samples ...")

print(f"\n📋 OPTION 3: Test first, then decide (recommended)")
print(f"\n   1️⃣  Test with 10 clusters:")
print(f"   python scripts/acquire_environmental_data.py \\")
print(f"       --phase 1 \\")
print(f"       --clusters {output_clusters} \\")
print(f"       --samples {samples_file} \\")
print(f"       --max-clusters 10")

print(f"\n   2️⃣  If successful, run full pipeline in parallel")

print(f"\n   3️⃣  Monitor progress:")
print(f"   python scripts/acquire_environmental_data.py --status")

print(f"\n⏱️  TIMING ESTIMATES:")
print(f"   • Sequential (Phase 1 then Phase 2): ~80.3 hours")
print(f"   • Parallel (both phases at once): ~62.7 hours (17.6h saved!)")
print(f"   • Max parallel (all 4 services): ~47.2 hours (slowest service)")

print(f"\n💾 DATABASE:")
print(f"   • Location: pangenome_env_data/pangenome_env.db")
print(f"   • SQLite WAL mode enabled for concurrent writes")
print(f"   • Each service tracks its own progress independently")
print(f"   • No conflicts - services write to different rows")

print(f"\n📊 Expected final data:")
print(f"   • Phase 1: ~{4789 * 19:,} observations, 19 features")
print(f"   • Phase 2: ~{4789 * 64:,} observations, 64 features")
print(f"   • Total: 83 environmental features × {len(target_samples):,} genomes")

In [ ]:
# Earth Engine Asset Survey and Production Service Tests
from env_agents.adapters import CANONICAL_SERVICES
from env_agents.core.models import RequestSpec, Geometry
import datetime

# Filter for TERRESTRIAL US locations (exclude ocean/coastal)
us_terrestrial = clusters_df[
    (clusters_df['center_lat'] > 30) & (clusters_df['center_lat'] < 48) &  # Continental US
    (clusters_df['center_lon'] > -120) & (clusters_df['center_lon'] < -70) &  # Avoid Pacific/Atlantic
    (clusters_df['center_lat'] != 27.52)  # Exclude the ocean cluster we just found
]

if len(us_terrestrial) > 0:
    test_cluster = us_terrestrial.iloc[0]
    print(f"🌎 Using terrestrial US location for data validation")
else:
    # Fallback to any cluster
    test_cluster = clusters_df.iloc[500]
    print(f"🌍 Using fallback test location")

print(f"📍 Test coordinates: {test_cluster['center_lat']:.4f}, {test_cluster['center_lon']:.4f}")

# Create TIGHT bounding box (0.01° ~= 1km) for fast SoilGrids queries
center_lat = test_cluster['center_lat']
center_lon = test_cluster['center_lon']
tight_bbox = [center_lat - 0.005, center_lon - 0.005, center_lat + 0.005, center_lon + 0.005]
test_geometry = Geometry(type="bbox", coordinates=[tight_bbox[1], tight_bbox[0], tight_bbox[3], tight_bbox[2]])

print(f"📦 Using tight bbox: {tight_bbox} (~1km²)")

# Test services configuration
test_services = {
    "NASA_POWER": {
        "time_range": ("2021-01-01", "2021-12-31"),
        "timeout": 60,
        "description": "NASA POWER Climate (proven fast)"
    },
    "SoilGrids": {
        "time_range": ("2021-01-01", "2021-12-31"),  # Ignored for static data
        "timeout": 120,
        "extra": {
            "max_pixels": 100,  # Drastically reduce for speed test
            "statistics": ["mean"],  # Only mean, not all stats
            "include_wrb": False  # Skip categorical WRB layer
        },
        "description": "SoilGrids (optimized: 100 pixels, mean only)"
    }
}

# Earth Engine assets
ee_assets = {
    "Google Embeddings": {
        "asset_id": "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL",
        "time_range": ("2021-01-01", "2021-12-31"),
        "timeout": 120
    },
    "MODIS NDVI": {
        "asset_id": "MODIS/061/MOD13Q1",
        "time_range": ("2021-01-01", "2021-12-31"),
        "timeout": 60
    },
    "SRTM Elevation": {
        "asset_id": "USGS/SRTMGL1_003",
        "time_range": ("2021-01-01", "2021-12-31"),
        "timeout": 60
    }
}

print("\n🧪 MULTI-SERVICE PRODUCTION TEST")
print("=" * 60)

all_results = {}

# Test regular services
for service_name, config in test_services.items():
    print(f"\n📦 Testing: {service_name}")
    print(f"   {config['description']}")
    
    try:
        adapter_class = CANONICAL_SERVICES[service_name]
        adapter = adapter_class()
        
        extra_params = {"timeout": config["timeout"]}
        if "extra" in config:
            extra_params.update(config["extra"])
        
        spec = RequestSpec(
            geometry=test_geometry,
            time_range=config["time_range"],
            variables=None,
            extra=extra_params
        )
        
        print(f"   🔍 Fetching...")
        start = datetime.datetime.now()
        result = adapter._fetch_rows(spec)
        elapsed = (datetime.datetime.now() - start).total_seconds()
        
        if result and len(result) > 0:
            vars_in_result = set(row.get('variable') for row in result if row.get('variable'))
            print(f"   ✅ Success: {len(result)} obs, {len(vars_in_result)} vars in {elapsed:.1f}s")
            all_results[service_name] = {
                'status': 'success',
                'obs': len(result),
                'vars': len(vars_in_result),
                'time_sec': elapsed
            }
        else:
            print(f"   ⚠️  No data")
            all_results[service_name] = {'status': 'no_data'}
            
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:80]}...")
        all_results[service_name] = {'status': 'error', 'error': str(e)[:200]}

# Test Earth Engine assets
print(f"\n🛰️ EARTH ENGINE ASSETS")
print("-" * 60)

EARTH_ENGINE = CANONICAL_SERVICES["EARTH_ENGINE"]
ee_results = {}

for asset_name, config in ee_assets.items():
    print(f"\n📦 {asset_name}")
    print(f"   Asset: {config['asset_id']}")
    
    try:
        adapter = EARTH_ENGINE(asset_id=config['asset_id'])
        caps = adapter.capabilities()
        print(f"   ✅ {len(caps.get('variables', []))} variables available")
        
        spec = RequestSpec(
            geometry=test_geometry,
            time_range=config['time_range'],
            variables=None,
            extra={"timeout": config['timeout']}
        )
        
        print(f"   🔍 Fetching...")
        start = datetime.datetime.now()
        result = adapter._fetch_rows(spec)
        elapsed = (datetime.datetime.now() - start).total_seconds()
        
        if result and len(result) > 0:
            vars_in_result = set(row.get('variable') for row in result if row.get('variable'))
            print(f"   ✅ Success: {len(result)} obs, {len(vars_in_result)} vars in {elapsed:.1f}s")
            ee_results[asset_name] = {
                'status': 'success',
                'obs': len(result),
                'vars': len(vars_in_result),
                'time_sec': elapsed
            }
        else:
            print(f"   ⚠️  No data")
            ee_results[asset_name] = {'status': 'no_data'}
            
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:80]}...")
        ee_results[asset_name] = {'status': 'error', 'error': str(e)[:200]}

# Combined summary
print("\n" + "=" * 60)
print("🎯 PRODUCTION READINESS SUMMARY")
print("=" * 60)

all_successful = {**{k: v for k, v in all_results.items() if v['status'] == 'success'},
                  **{k: v for k, v in ee_results.items() if v['status'] == 'success'}}

if all_successful:
    print(f"\n✅ WORKING SERVICES ({len(all_successful)}):")
    total_vars = 0
    for svc, data in all_successful.items():
        total_vars += data['vars']
        print(f"   • {svc}: {data['vars']} vars, {data['obs']} obs ({data['time_sec']:.1f}s)")
    
    print(f"\n📊 Total Environmental Features: {total_vars}")
    print(f"⚡ Average query time: {sum(d['time_sec'] for d in all_successful.values())/len(all_successful):.1f}s")
    print(f"📦 Estimated time for 4,789 clusters:")
    print(f"   • Per service: ~{4789 * sum(d['time_sec'] for d in all_successful.values())/len(all_successful)/3600:.1f} hours")
    print(f"   • All {len(all_successful)} services: ~{len(all_successful) * 4789 * sum(d['time_sec'] for d in all_successful.values())/len(all_successful)/3600:.1f} hours")
    
    if "Google Embeddings" in all_successful:
        print(f"\n🧬 Google Satellite Embeddings: {all_successful['Google Embeddings']['vars']} dimensions CONFIRMED")
    
    print(f"\n💡 OPTIMIZATION RECOMMENDATIONS:")
    if "SoilGrids" in all_successful:
        if all_successful["SoilGrids"]['time_sec'] > 60:
            print(f"   ⚠️  SoilGrids took {all_successful['SoilGrids']['time_sec']:.0f}s - consider max_pixels=50 or skip for global dataset")
        else:
            print(f"   ✅ SoilGrids optimized successfully ({all_successful['SoilGrids']['time_sec']:.1f}s)")
    print(f"   📍 Use tight bboxes (~0.01°) for faster queries")
    print(f"   🚀 Process services in parallel where possible")

else:
    print("\n⚠️  No services returned data at test location")
    print("   This may indicate:")
    print("   • Test location has sparse data coverage")
    print("   • Service authentication issues")
    print("   • Network/timeout problems")

{'regular_services': all_results, 'earth_engine': ee_results}